In [1]:
from dotenv import load_dotenv
import os 
load_dotenv()

True

In [17]:
import os
import langchain_mcp_adapters.sessions as sessions

_orig_stdio_client = sessions.stdio_client

def _patched_stdio_client(server, errlog=None):
    # real file object with a fileno() instead of the notebook's fake stderr
    return _orig_stdio_client(server, errlog=open("mcp_server.log", "w"))

sessions.stdio_client = _patched_stdio_client

In [20]:
from langchain_mcp_adapters.client import MultiServerMCPClient

api_key = os.getenv("CONTEXT7_API_KEY")

client = MultiServerMCPClient(
    {
        "filesystem": {
            "transport": "stdio",
            "command": "cmd",
            "args": [
                "/c",
                "npx",
                "-y",
                "@modelcontextprotocol/server-filesystem",
                r"C:\Users\pc\Desktop\lang_work",
            ],
        },
        "context7": {
            "transport": "streamable_http",
            "url": "https://mcp.context7.com/mcp",
            "headers": {"Authorization": f"Bearer {api_key}"},
        },
    }
)

tools = await client.get_tools()
print([tool.name for tool in tools])

['read_file', 'read_text_file', 'read_media_file', 'read_multiple_files', 'write_file', 'edit_file', 'create_directory', 'list_directory', 'list_directory_with_sizes', 'directory_tree', 'move_file', 'search_files', 'get_file_info', 'list_allowed_directories', 'resolve-library-id', 'query-docs']


In [ ]:
### basic prompt usage
from pprint import pprint

from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv

load_dotenv()

model = ChatOpenAI(
    model="meta/muse-glimmer-30b",
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("NVIDIA_API_KEY"),
    temperature=0.2,
    max_tokens=500,
    reasoning_effort="medium"
)

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt="You are a helpful assistant. "
)

res = await agent.ainvoke({
    "messages":[
          {
                    "role": "user",
                    "content": "Use Context7 to find the official LangChain Python documentation for create_agent. Explain how to use it with a chat model and tools. Include a working code example.",
                }
        ]
})

pprint(res)


{'messages': [HumanMessage(content='create file on langwork called test.txt', additional_kwargs={}, response_metadata={}, id='dc832fce-6ab0-414d-a27f-fb8161f48ede'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 226, 'prompt_tokens': 1457, 'total_tokens': 1683, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'meta/muse-glimmer-30b', 'system_fingerprint': None, 'id': '9b6a4002686713f0', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a15031-615d-7a53-a1e9-a45ffd65fe85-0', tool_calls=[{'name': 'resolve-library-id', 'args': {'query': 'create file test.txt', 'libraryName': 'LangWork'}, 'id': 'chatcmpl-tool-887788ad3050e125', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 1457, 'output_tokens': 2

In [ ]:
agent = create_agent(
    model=model,
    tools=tools,
    system_prompt="You are a helpful assistant. "
)
res = await agent.ainvoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "use filesystem tool to Create a file named test.txt in the allowed workspace "
                    "containing: Hello from my LangChain agent."
                ),
            }
        ]
    }
)

print(res["messages"][-1].content)


Created the file `test.txt` in the allowed workspace at `C:\Users\pc\Desktop\lang_work\test.txt` with the contents:

Hello from my LangChain agent.
